# Energy pricing calibration — 02 traction energy prices

Single source of truth for every calibrated energy-pricing value, and the
generator for `ENERGY_PRICING_CALIBRATION.md`. Run
`01_source_extraction.ipynb` first.

Scope is the **price of driving energy**: the electricity a train draws while
running, and the charges an infrastructure manager levies for supplying it
(catenary and traction power-supply installations). Consumption modelling
(kWh per train-km) belongs to `models/energy/`; stabling and pre-heating
energy belongs to the facility domain; track access belongs to
`models/infrastructure/tac/`.

Outputs, all generated and none hand-edited: `data/energy_prices.csv`,
`data/energy_price_modes.csv`, `seed/track_energy.csv`,
`seed/track_energy_default.csv`, `seed/sources.csv`,
`ENERGY_PRICING_CALIBRATION.md` and `energy_price_by_country.svg`.

In [ ]:
# Energy pricing calibration — traction energy prices and supply charges
#
# Single source of truth for every calibrated value. The CSVs under `data/`,
# the seed CSVs under `seed/` and the document ENERGY_PRICING_CALIBRATION.md
# are generated artifacts: re-run this notebook (after 01) to regenerate them,
# never hand-edit.
#
# Scope is the price of driving energy only. What is deliberately elsewhere:
# consumption in kWh (models/energy/), stabling and pre-heating energy
# (facility domain), and the track access charge itself
# (models/infrastructure/tac/) — which excludes every supply-equipment charge
# per country precisely so this domain can price them without double-counting.

import csv
from dataclasses import dataclass, asdict
from pathlib import Path


def _resolve_data_dir() -> Path:
    here = Path.cwd()
    for cand in (
        here / "data",
        here / "backend/models/infrastructure/energy_pricing/calib/data",
    ):
        if cand.parent.exists():
            cand.mkdir(exist_ok=True)
            return cand
    raise RuntimeError(f"cannot locate the calib data directory from {here}")


DATA_DIR = _resolve_data_dir()
print(f"data directory: {DATA_DIR}")

# Date this calibration was last reviewed end to end. Held here rather than
# taken from the register so 02 stands alone: the notebooks share files, not
# Python state, and either can be re-run without the other in memory.
CALIBRATION_REVIEWED = "2026-08-17"

# --- provenance vocabulary -------------------------------------------------
# How much evidence stands behind a value. Not a quality judgement — a
# well-argued ASSUMED and a mis-transcribed SOURCED are both possible; the
# status says which kind of thing the reader is looking at.
SOURCED = "sourced"  # named document, named locator
DERIVED = "derived"  # arithmetic on other values, formula in the note
BENCHMARK = "benchmark"  # pan-European statistic standing in for a country
ASSUMED = "assumed"  # judgement, with a mandatory low/high band
NOT_LEVIED = "not_levied"  # positively documented as absent, not unknown
MISSING = "missing"  # nothing read yet — never an estimate
NO_RAILWAY = "no_railway"  # positively documented as having no network

# --- price basis and conversion -------------------------------------------
# The evaluation year of the target network. Every monetary value is carried
# here before it reaches the database.
TARGET_YEAR = 2032

# Electricity is escalated at nominal HICP, NOT at the 3%/yr the TAC
# calibration uses. Track access rises in real terms because Directive
# 2012/34 Art.31-32 pushes IMs towards full cost recovery against a growing
# renewal burden; a traded commodity has no such mechanism. European wholesale
# power forwards run flat to falling in real terms beyond 2027 as renewable
# capacity displaces gas at the margin, so carrying the 2025 commodity price
# forward at HICP is the neutral choice: constant real price, nominal drift
# only. The band brackets the two directions the argument can fail in — 0%/yr
# if real prices fall as fast as nominal inflation rises, 3%/yr if grid
# reinforcement and levy growth outpace HICP.
ENERGY_ESCALATION_PER_YEAR = 0.02
ENERGY_ESCALATION_LOW = 0.00
ENERGY_ESCALATION_HIGH = 0.03

# Deviations from that rate, each with a mandatory reason. A parameter-level
# override outranks a country-level one: a statutory tax rate does not move
# with a national tariff's escalation, whichever country levies it.
ESCALATION_OVERRIDE_PARAMETER: dict[str, tuple[float, str]] = {
    "rail_electricity_tax": (
        0.0,
        "Electricity excise is a statutory rate per kWh, not an indexed "
        "tariff: it moves when a legislature moves it and not otherwise. "
        "Germany is the direct evidence — the StromStG §9(2) rail rate reads "
        "11.42 EUR/MWh in the 2016 CE Delft database and 11.42 EUR/MWh in DB "
        "InfraGO's 2027 price list, unchanged across a decade that included "
        "the 2022 energy crisis. Austria's Elektrizitätsabgabe likewise sits "
        "at its long-standing 15.00 EUR/MWh once the temporary 2022-2024 "
        "reduction expired. COUNTER-ARGUMENT, deliberately recorded: a "
        "nominal freeze is a real-terms cut, and the ETS2 and energy-taxation "
        "directive files both point at higher minimum rates this decade; if "
        "either lands, this becomes the low end rather than the estimate. The "
        "exposure is small — the largest rail excise in the calibration is "
        "1.4% of a working price.",
    ),
}

ESCALATION_OVERRIDE_COUNTRY: dict[str, tuple[float, str]] = {
    "SK": (
        0.0,
        "The Slovak supply-equipment charge is component U4 of Measure "
        "2/2018, frozen with the rest of that instrument since 2019. Same "
        "deviation, same reason and the same recorded counter-argument as the "
        "Slovak track access charges: a fourteen-year freeze is implausible "
        "and a revision would likely catch up at once, so treat zero as the "
        "low end of a real 0-2%/yr range.",
    ),
}


def escalation_rate(country_code: str, parameter: str = "") -> float:
    """The escalation rate for one value — HICP unless a documented
    deviation applies. Parameter overrides outrank country overrides."""
    if parameter in ESCALATION_OVERRIDE_PARAMETER:
        return ESCALATION_OVERRIDE_PARAMETER[parameter][0]
    override = ESCALATION_OVERRIDE_COUNTRY.get(country_code)
    return ENERGY_ESCALATION_PER_YEAR if override is None else override[0]


# ECB reference rates, snapshot date below — deliberately the same snapshot
# the TAC calibration pins, so both infrastructure domains reach EUR on
# identical terms and a scenario can repin one date rather than two.
FX_SNAPSHOT = "2026-08-11"
FX_TO_EUR = {
    "EUR": 1.0,
    "CHF": 1.064,
    "CZK": 1 / 24.5,
    "DKK": 1 / 7.46,
    "GBP": 1.20,
    "HUF": 1 / 396.0,
    "NOK": 1 / 11.7,
    "PLN": 1 / 4.25,
    "RON": 1 / 5.08,
    "SEK": 1 / 11.2,
}


def to_eur(value: float, currency: str) -> float:
    """Native currency to EUR at the pinned FX snapshot."""
    return value * FX_TO_EUR[currency]


def escalate(value: float, basis_year: int, rate: float) -> float:
    """Carry a value from its document price basis to the evaluation year."""
    return value * (1.0 + rate) ** (TARGET_YEAR - basis_year)


def to_model_value(
    value: float, currency: str, basis_year: int, country_code: str, parameter: str = ""
) -> float:
    """The number the database receives: EUR, at the evaluation year.

    Both conversions happen exactly once, here. The cost model never sees a
    native currency or a price basis.
    """
    return escalate(
        to_eur(value, currency), basis_year, escalation_rate(country_code, parameter)
    )

## Value record

One row per country and parameter, in the currency and price basis the source
document uses. The EUR and evaluation-year forms are derived on demand, never
stored — so every row in `data/` stays checkable line by line against the
document it came from.

In [ ]:
# --- value record ----------------------------------------------------------
@dataclass
class SV:
    """One calibrated value with its provenance.

    `value` is always as published, in `currency` at `basis_year`. A None
    value with status NOT_LEVIED is a tariff fact — this country does not
    charge this — and must not be confused with MISSING, which is an
    admission that nothing has been read yet.
    """

    country_code: str
    parameter: str
    value: float | None
    unit: str
    status: str
    source_id: str = ""
    locator: str = ""
    currency: str = "EUR"
    basis_year: int | None = None
    note: str = ""
    low: float | None = None
    high: float | None = None

    def __post_init__(self):
        if self.status == ASSUMED and (self.low is None or self.high is None):
            raise ValueError(
                f"{self.country_code}.{self.parameter}: ASSUMED needs a band"
            )
        if self.status in (SOURCED, DERIVED, BENCHMARK) and not self.source_id:
            raise ValueError(
                f"{self.country_code}.{self.parameter}: {self.status} needs a source"
            )
        if self.status in (NOT_LEVIED, DERIVED) and not self.note:
            raise ValueError(
                f"{self.country_code}.{self.parameter}: {self.status} must state why"
            )
        if self.value is not None and self.basis_year is None:
            raise ValueError(
                f"{self.country_code}.{self.parameter}: a value needs a price basis"
            )

    @property
    def eur(self) -> float | None:
        return None if self.value is None else to_eur(self.value, self.currency)

    @property
    def model_value(self) -> float | None:
        """EUR at the evaluation year — what the seed export writes."""
        if self.value is None:
            return None
        return to_model_value(
            self.value,
            self.currency,
            self.basis_year,
            self.country_code,
            self.parameter,
        )


SV_FIELDS = [
    "country_code",
    "parameter",
    "value",
    "unit",
    "status",
    "source_id",
    "locator",
    "currency",
    "basis_year",
    "note",
    "low",
    "high",
]

# The four parameters the database actually holds. Every country carries all
# four so a NULL is explicit rather than absent — a country that levies no
# catenary charge and a country nobody has checked must not look alike.
# Component-level rows (Eurostat columns, overrides, taxes) are emitted only
# where the derivation mode uses them, since a Eurostat breakdown for a
# country priced from an all-in tariff would be noise, not provenance.
DB_PARAMETERS = [
    "working_price_day",
    "working_price_night",
    "catenary_eur_train_km",
    "catenary_eur_gross_tonne_km",
]

COUNTRIES = [
    "AT",
    "BE",
    "BG",
    "CH",
    "CY",
    "CZ",
    "DE",
    "DK",
    "EE",
    "ES",
    "FI",
    "FR",
    "GR",
    "HR",
    "HU",
    "IE",
    "IT",
    "LT",
    "LU",
    "LV",
    "MT",
    "NL",
    "NO",
    "PL",
    "PT",
    "RO",
    "SE",
    "SI",
    "SK",
    "UK",
]

# CY and MT have no railway at all — a positive fact, distinct from an
# uncalibrated country. UK is the register's own code for Great Britain;
# db/dev/seed.py maps it to the ISO code GB on the way in, once.
NO_RAILWAY_COUNTRIES = {"CY", "MT"}

values: list[SV] = []

## Mode (a) — the Eurostat benchmark

Nine price components per country, band IE (20,000–69,999 MWh/a), 2025. This
is the base for every country without a published traction-energy tariff, and
the layer the two override modes below cut into.

In [ ]:
# --- Mode (a): Eurostat band IE components ---------------------------------
# Band IE is the rail-relevant consumption band: a night-train operation at
# 15-25 kWh/train-km over 1-3 million train-km a year draws 20-70 GWh, which
# is band IE exactly. Two independent IM tariffs validate the choice (HU and
# SE, below) and neither matches the cheaper-looking band IA that an earlier
# draft used.
EUROSTAT_LOCATOR = "band IE, 2025, nrg_pc_205_c"
EUROSTAT_BASIS = 2025

COMPONENTS = (
    "energy_supply",
    "network",
    "vat",
    "renewable",
    "capacity",
    "environmental",
    "nuclear",
    "other",
)
NON_VAT = tuple(c for c in COMPONENTS if c != "vat")

EUROSTAT_IE = {
    "AT": (0.1112, 0.0292, 0.0322, 0.0047, 0.0000, 0.0135, 0.0000, 0.0024),
    "BE": (0.0968, 0.0169, 0.0279, 0.0113, 0.0008, 0.0072, 0.0000, 0.0000),
    "BG": (0.1108, 0.0219, 0.0251, 0.0000, 0.0000, 0.0010, 0.0000, -0.0082),
    "CZ": (0.1114, 0.0414, 0.0351, 0.0129, 0.0000, 0.0012, 0.0000, 0.0001),
    "DE": (0.0979, 0.0375, 0.0321, 0.0017, 0.0097, 0.0205, 0.0000, 0.0014),
    "DK": (0.0912, 0.0261, 0.0534, 0.0000, 0.0000, 0.0011, 0.0000, 0.0000),
    "EE": (0.0811, 0.0209, 0.0269, 0.0084, 0.0000, 0.0018, 0.0000, 0.0000),
    "ES": (0.0896, 0.0100, 0.0222, 0.0007, 0.0002, 0.0035, 0.0000, 0.0050),
    "FI": (0.0375, 0.0134, 0.0131, 0.0000, 0.0001, 0.0005, 0.0000, 0.0000),
    "FR": (0.0680, 0.0175, 0.0141, 0.0000, 0.0012, 0.0057, 0.0000, 0.0000),
    "GR": (0.1344, 0.0112, 0.0093, 0.0028, 0.0000, 0.0020, 0.0000, 0.0054),
    "HR": (0.1034, 0.0171, 0.0172, 0.0111, 0.0000, 0.0005, 0.0000, 0.0000),
    "HU": (0.1175, 0.0352, 0.0415, 0.0135, 0.0000, 0.0010, 0.0000, 0.0039),
    "IE": (0.1529, 0.0474, 0.0129, 0.0019, 0.0001, 0.0003, 0.0000, 0.0040),
    "IT": (0.1223, 0.0166, 0.0158, 0.0147, 0.0083, 0.0015, 0.0000, 0.0016),
    "LT": (0.0940, 0.0258, 0.0250, 0.0002, 0.0000, 0.0000, 0.0000, 0.0002),
    "LU": (0.1036, 0.0110, 0.0092, 0.0003, 0.0000, 0.0000, 0.0000, 0.0002),
    "LV": (0.0871, 0.0111, 0.0214, 0.0000, 0.0000, 0.0000, 0.0000, 0.0039),
    "NL": (0.0959, 0.0200, 0.0271, 0.0000, 0.0000, 0.0132, 0.0000, 0.0000),
    "NO": (0.0516, 0.0053, 0.0163, 0.0000, 0.0000, 0.0082, 0.0000, 0.0000),
    "PL": (0.0775, 0.0278, 0.0373, 0.0022, 0.0099, 0.0446, 0.0000, 0.0003),
    "PT": (0.0814, 0.0146, 0.0235, 0.0079, 0.0005, 0.0005, 0.0000, 0.0011),
    "RO": (0.1128, 0.0311, 0.0313, 0.0128, 0.0000, 0.0005, 0.0000, 0.0000),
    "SE": (0.0530, 0.0161, 0.0174, 0.0000, 0.0000, 0.0005, 0.0000, 0.0000),
    "SI": (0.1130, 0.0126, 0.0290, 0.0050, 0.0001, 0.0009, 0.0000, 0.0000),
    "SK": (0.1135, 0.0344, 0.0329, 0.0111, 0.0096, 0.0013, 0.0033, 0.0000),
}

# The EU-27 row is a reference line for the document only — never a country
# value, never seeded.
EUROSTAT_EU27 = (0.0946, 0.0235, 0.0247, 0.0044, 0.0041, 0.0102, 0.0000, 0.0012)

eurostat: dict[str, dict[str, SV]] = {}
for cc, row in EUROSTAT_IE.items():
    eurostat[cc] = {
        name: SV(
            cc,
            f"eurostat_{name}",
            v,
            "EUR/kWh",
            BENCHMARK,
            "EUROSTAT-NRG-PC-205-C",
            EUROSTAT_LOCATOR,
            "EUR",
            EUROSTAT_BASIS,
            "Eurostat price component, non-household band IE",
        )
        for name, v in zip(COMPONENTS, row)
    }
print(f"{len(eurostat)} countries on the Eurostat basis")

## Mode (b) — full national or IM tariff

Five countries publish a traction-energy price of their own, which replaces
the Eurostat row entirely rather than any single component of it. Two of them
publish a day and a night rate.

In [ ]:
# --- Mode (b): full override — an all-in published tariff ------------------
# The value replaces the whole Eurostat stack, including the tax and VAT
# treatment, because these tariffs are quoted as what the RU pays.
CH_NIGHT = SV(
    "CH",
    "im_tariff_night",
    0.078,
    "CHF/kWh",
    SOURCED,
    "CH-NZV",
    "Art.20a transitional",
    "CHF",
    2027,
    "22:00-06:00 rate, 40% below the 0.13 base. All-in: Switzerland levies no "
    "electricity excise on rail and no separate supply-equipment charge",
)

IM_TARIFF: dict[str, dict[str, SV]] = {
    "CH": {
        "day": SV(
            "CH",
            "im_tariff_day",
            0.130,
            "CHF/kWh",
            SOURCED,
            "CH-NZV",
            "Art.20a transitional",
            "CHF",
            2027,
            "Base rate outside the night band",
        ),
        "night": CH_NIGHT,
    },
    "HR": {
        "day": SV(
            "HR",
            "im_tariff_day",
            0.1417,
            "EUR/kWh",
            DERIVED,
            "HR-NS-2027",
            "ch.5.4 item 40",
            "EUR",
            2027,
            "Day tariff (VT) 0.1285 + renewables levy 0.0132; excise printed "
            "as 0.000000, which supersedes the CE Delft PPS figure for HR",
        ),
        "night": SV(
            "HR",
            "im_tariff_night",
            0.0958,
            "EUR/kWh",
            DERIVED,
            "HR-NS-2027",
            "ch.5.4 item 40",
            "EUR",
            2027,
            "Night tariff (NT) 0.0826 + renewables levy 0.0132",
        ),
    },
    "HU": {
        "day": SV(
            "HU",
            "im_tariff_day",
            67.4,
            "HUF/kWh",
            SOURCED,
            "HU-NS-2627",
            "Annex 5.2-6",
            "HUF",
            2027,
            "All-in: 49.0 energy + 10.2 system + 0.4 excise + 7.8 funds. One "
            "of the two band-IE validations — 67.4 HUF is 0.170 EUR against "
            "Eurostat band IE HU at 0.1711",
        )
    },
    "SE": {
        "day": SV(
            "SE",
            "im_tariff_day",
            0.7156,
            "SEK/kWh",
            DERIVED,
            "SE-NS-2027",
            "§7.3.11",
            "SEK",
            2027,
            "Worked example: energy 0.6075 + grid 0.1081. The second band-IE "
            "validation — 0.064 EUR against Eurostat band IE SE at 0.0696",
        )
    },
    "UK": {
        "day": SV(
            "UK",
            "im_tariff_day",
            0.238480,
            "GBP/kWh",
            SOURCED,
            "DESNZ-T341",
            "Table 3.4.1, Large band",
            "GBP",
            2025,
            "Excl. CCL — rail traction is CCL-exempt, so that is the working "
            "series. The Large band is defined as band IE is",
        )
    },
}
print(f"{len(IM_TARIFF)} countries on a full override")

## Mode (c) — traction-network override

Austria, Germany and France each run a traction-current network physically
separate from the public grid and charge for its use. That tariff replaces
**only** the Eurostat network column: the current itself is still procured on
the market, so the commodity and tax columns stay on the benchmark basis.

In [ ]:
# --- Mode (c): network-column override ------------------------------------
NETWORK_OVERRIDE: dict[str, dict[str, SV]] = {
    "AT": {
        "day": SV(
            "AT",
            "network_override_day",
            0.05240,
            "EUR/kWh",
            SOURCED,
            "AT-SNNB-2026",
            "Tab.59 p.114",
            "EUR",
            2026,
            "Bahnstromnetz Hochtarif 06:00-22:00, 52.40 EUR/MWh. Single-part "
            "tariff billed on energy drawn, so no demand assumption is needed",
        ),
        "night": SV(
            "AT",
            "network_override_night",
            0.04367,
            "EUR/kWh",
            SOURCED,
            "AT-SNNB-2026",
            "Tab.59 p.114",
            "EUR",
            2026,
            "Bahnstromnetz Niedertarif 22:00-06:00, 43.67 EUR/MWh",
        ),
    },
    "DE": {
        "day": SV(
            "DE",
            "network_override_day",
            0.0844,
            "EUR/kWh",
            ASSUMED,
            "DE-DBE-NETZ-2026",
            "Hochspannung, <2,500 h/a",
            "EUR",
            2026,
            "Two-part tariff: Arbeitspreis 0.0721 plus Leistungspreis 23.35 "
            "EUR/kWa at an assumed 1,900 h/a Benutzungsdauer. The band matters "
            "more than the point estimate and is narrow — 1,500 h/a gives "
            "0.0877 and 2,400 h/a gives 0.0818, because the Leistungspreis is "
            "small below 2,500 h/a. A night-only operation cannot reach the "
            ">=2,500 h/a system, which needs about 4,250 running hours a year. "
            "Statutory levies (KWKG, Offshore-Netzumlage, StromNEV §19(2)) are "
            "deliberately not added: they are non-network levies already "
            "carried in the Eurostat tax columns this override leaves intact",
            low=0.0818,
            high=0.0877,
        )
    },
    "FR": {
        "day": SV(
            "FR",
            "network_override_day",
            0.0315,
            "EUR/kWh",
            DERIVED,
            "FR-DRR-A512",
            "loss formula §2.1",
            "EUR",
            2027,
            "RCTE-A = purchase price x loss rate / (1 - loss rate) = 0.0680 x "
            "0.134/0.866 = 0.0105, plus RCTE-B 0.0210 (TURPE pass-through, "
            "indexed from 0.02003). The formula is validated by back-solving "
            "the published 2024 tariff: 0.02912 x 0.864/0.136 = 0.1850 EUR/kWh "
            "implied purchase price, against a published 2024 RFE of 0.18653. "
            "Replaces the crisis-era 2024 network total of 0.04915; SNCF "
            "Réseau publishes the real 2027 values in December 2026",
        )
    },
}

# Night bands, minute of day, wrapping midnight. Only three countries price
# electricity by clock time; everyone else charges one rate around the clock,
# which the model reads as "no band" rather than as a missing band.
NIGHT_BANDS: dict[str, SV] = {
    "AT": SV(
        "AT",
        "night_band",
        22 * 60,
        "minute of day",
        SOURCED,
        "AT-SNNB-2026",
        "Tab.59 p.114",
        "EUR",
        2026,
        "Niedertarif 22:00-06:00, stated in the tariff table",
    ),
    "CH": SV(
        "CH",
        "night_band",
        22 * 60,
        "minute of day",
        SOURCED,
        "CH-NZV",
        "Art.20a",
        "CHF",
        2027,
        "Reduced rate 22:00-06:00, stated in the ordinance",
    ),
    "HR": SV(
        "HR",
        "night_band",
        22 * 60,
        "minute of day",
        ASSUMED,
        "HR-NS-2027",
        "ch.5.4 item 40",
        "EUR",
        2027,
        "The network statement prices NT and VT without printing the hours. "
        "22:00-06:00 is the standard Croatian low-tariff window and matches "
        "both neighbours that do publish theirs (AT, CH). The band matters "
        "less than it looks: HR day and night differ by 0.046 EUR/kWh, so a "
        "one-hour error moves a Croatian leg by fractions of a percent",
        low=21 * 60,
        high=23 * 60,
    ),
}
NIGHT_BAND_END_MIN = 6 * 60  # 06:00 in all three countries

for cc, band in NIGHT_BANDS.items():
    assert cc in IM_TARIFF or cc in NETWORK_OVERRIDE, (
        f"{cc} has a night band but no banded tariff to apply it to"
    )
print(f"{len(NETWORK_OVERRIDE)} network overrides, {len(NIGHT_BANDS)} night bands")

## Rail-specific electricity tax

The national electricity excise at the rate rail actually pays. Eurostat's
Environmental taxes column carries the **standard** rate, so the two are the
same tax at different rates and must not be added — the rail rate replaces the
column.

In [ ]:
# --- Rail electricity tax --------------------------------------------------
# CE Delft's values are PPS-adjusted (purchasing-power corrected), so nominal
# rates differ by the national price level. The largest entry in the table is
# 1.4% of a typical working price, which is why the PPS-versus-nominal
# question is documented rather than corrected — except for AT and DE, where
# the nominal rate is independently published and is used instead.
_CD, _CDL = "CE-DELFT-4K83", "Rail_Energy taxes_level"
_CD_NOTE = "PPS-adjusted 2016 rate; nominal differs by the national price level"

RAIL_TAX: dict[str, SV] = {}
for cc, v in {
    "BG": 0.002145,
    "DK": 0.000372,
    "EE": 0.006102,
    "ES": 0.005677,
    "FR": 0.000457,
    "GR": 0.000609,
    "HU": 0.001691,
    "LT": 0.000848,
    "LU": 0.000414,
    "NL": 0.002332,
    "PL": 0.008234,
    "RO": 0.001039,
    "SI": 0.003968,
}.items():
    RAIL_TAX[cc] = SV(
        cc,
        "rail_electricity_tax",
        v,
        "EUR/kWh",
        SOURCED,
        _CD,
        _CDL,
        "EUR",
        2016,
        _CD_NOTE,
    )

# Denmark is the one country whose descriptive column says "exempt" while the
# database carries a rate and reports 2016 revenue. The number is the more
# specific evidence and is adopted; either way it is 0.3% of the DK price.
RAIL_TAX["DK"] = SV(
    "DK",
    "rail_electricity_tax",
    0.000372,
    "EUR/kWh",
    SOURCED,
    _CD,
    _CDL,
    "EUR",
    2016,
    _CD_NOTE + ". The database rate supersedes the sheet's own 'exempt' label, "
    "which contradicts both the value and the reported revenue",
)

for cc in (
    "BE",
    "CZ",
    "FI",
    "HR",
    "IE",
    "IT",
    "LV",
    "NO",
    "PT",
    "SE",
    "SK",
    "UK",
    "CH",
):
    RAIL_TAX[cc] = SV(
        cc,
        "rail_electricity_tax",
        0.0,
        "EUR/kWh",
        NOT_LEVIED,
        _CD,
        _CDL,
        "EUR",
        2016,
        "Rail traction exempt or no electricity excise levied",
    )

# The two rates large enough to matter are both published at nominal and both
# confirmed twice, so they do not use the PPS figures.
RAIL_TAX["AT"] = SV(
    "AT",
    "rail_electricity_tax",
    0.0150,
    "EUR/kWh",
    SOURCED,
    "APS-STROMSTEUER",
    "EU comparison table",
    "EUR",
    2016,
    "Elektrizitätsabgabe at the full rate — Austria grants rail no carve-out, "
    "and the temporary 2022-2024 reduction to 1 EUR/MWh expired 31 Dec 2024. "
    "Corroborated by CE-DELFT-4K83 at 13.79 EUR/MWh PPS-adjusted",
)
RAIL_TAX["DE"] = SV(
    "DE",
    "rail_electricity_tax",
    0.01142,
    "EUR/kWh",
    SOURCED,
    "DE-APS-2027",
    "StromStG rates footnote",
    "EUR",
    2027,
    "StromStG §9(2) reduced rail rate, available on presentation of an "
    "Erlaubnisschein, read at the 2027 price basis. Identical to the 2016 CE "
    "Delft figure — the direct evidence that a statutory rate does not "
    "escalate with a tariff",
)

# Poland is the one country where the reconciliation is not applied. Its
# Eurostat Environmental column (0.0446 EUR/kWh) is around 37 times the Polish
# electricity excise, so it evidently bundles certificate-of-origin and
# cogeneration obligations that are not the excise at all. Subtracting it
# would credit Poland with levies it really pays; splitting it would be a
# guess. The un-reconciled total is used, which overstates rather than
# understates Polish energy cost.
NO_TAX_RECONCILIATION = {"PL"}

# VAT is only a cost where the operator's own output is VAT-EXEMPT, since
# exemption removes the right to deduct input VAT. Zero-rating preserves it.
VAT_RECOVERABLE = {cc: True for cc in set(EUROSTAT_IE) | set(IM_TARIFF)}
VAT_RECOVERABLE["DK"] = False

VAT_FLAG = SV(
    "DK",
    "vat_non_deductible",
    1.0,
    "flag",
    SOURCED,
    "SKAT-DK-VAT",
    "Services exempt from VAT",
    "EUR",
    2024,
    "Danish passenger transport is VAT-exempt, so input VAT on electricity "
    "cannot be reclaimed and is a real cost",
)
print(
    f"{len(RAIL_TAX)} rail tax entries, VAT non-deductible in "
    f"{sum(1 for v in VAT_RECOVERABLE.values() if not v)} country"
)

## Electric supply equipment charges

What an infrastructure manager charges for *use of the catenary and the
traction power-supply installations*, as distinct from the energy drawn
through them. Every one of these is a line the TAC calibration recorded as
"excluded (energy)" — so this table is the other half of that decision, and
the reason the two domains together neither double-count nor drop anything.

Units are the IM's own: most charge per train-km, three per gross-tonne-km,
and Belgium per MWh — the last of which is a price per unit of energy and
therefore joins the Belgian working price rather than becoming a column.

In [ ]:
# --- Electric supply equipment ---------------------------------------------
# Per train-km. Not converted to a per-kWh equivalent, deliberately: dividing
# by an assumed consumption would bake models/energy's flat 28 kWh/km
# placeholder into an infrastructure charge and silently move every one of
# these numbers when that model is calibrated.
CATENARY_TRAIN_KM: dict[str, SV] = {
    "FR": SV(
        "FR",
        "catenary_eur_train_km",
        0.291,
        "EUR/train-km",
        SOURCED,
        "FR-DRR-2027-A52",
        "App.5.2.2 RCE",
        "EUR",
        2027,
        "Redevance de circulation électrique — use of the catenary, separate "
        "from the RCTE components that price transport of the energy itself",
    ),
    "HR": SV(
        "HR",
        "catenary_eur_train_km",
        0.10,
        "EUR/train-km",
        SOURCED,
        "HR-NS-2027",
        "§5.3 items 4-18",
        "EUR",
        2027,
        "Electric-traction surcharge, flat and outside the T x Li x Cvlkm "
        "factor chain that scales the Croatian track charge",
    ),
    "HU": SV(
        "HU",
        "catenary_eur_train_km",
        110.0,
        "HUF/train-km",
        SOURCED,
        "HU-NS-2627",
        "Annex 5.2-6",
        "HUF",
        2027,
        "Catenary use per electric train-km",
    ),
    "IT": SV(
        "IT",
        "catenary_eur_train_km",
        0.241,
        "EUR/train-km",
        ASSUMED,
        "IT-LISTINO",
        "Component TA3",
        "EUR",
        2027,
        "RFI publishes two rates — 0.241 on the conventional network and 0.482 "
        "on high speed — and which applies depends on the line a leg runs on. "
        "The model does not resolve Italian line class yet, and a night train "
        "on the Basic segment runs the conventional network by default, so the "
        "lower rate is the working value with the pair as the band",
        low=0.241,
        high=0.482,
    ),
    "LT": SV(
        "LT",
        "catenary_eur_train_km",
        0.1709,
        "EUR/train-km",
        SOURCED,
        "LT-LTG-2627",
        "§5.3.2",
        "EUR",
        2027,
        "Contact-network fee. Note the shape: Lithuania charges track access "
        "purely per gross-tonne-km and the catenary purely per train-km",
    ),
    "LU": SV(
        "LU",
        "catenary_eur_train_km",
        0.2583,
        "EUR/train-km",
        SOURCED,
        "LU-NS-2027",
        "§5.3.2 c_E",
        "EUR",
        2026,
        "Electric supply component, flat — none of the train-length or "
        "category factors the Luxembourg base charge carries apply to it",
    ),
    "LV": SV(
        "LV",
        "catenary_eur_train_km",
        0.15,
        "EUR/train-km",
        SOURCED,
        "LV-NS-2027",
        "§5.2",
        "EUR",
        2026,
        "Electric-traction supply-equipment charge for international "
        "passenger services within the EEA",
    ),
    "PL": SV(
        "PL",
        "catenary_eur_train_km",
        0.29,
        "PLN/train-km",
        SOURCED,
        "PL-PLK-A91",
        "Annex 9.1",
        "PLN",
        2027,
        "Electric-traction component, outside the WM (mass) and WK (line "
        "category) factors that scale the base rate",
    ),
    "RO": SV(
        "RO",
        "catenary_eur_train_km",
        0.676,
        "RON/train-km",
        SOURCED,
        "RO-CFR-A25",
        "Annex 26.a Ttse",
        "RON",
        2024,
        "Electrification component. Rates valid from 1 Mar 2024, the longest "
        "escalation path in the domain outside the statutory taxes",
    ),
}

# Per gross-tonne-km. Three IMs charge the supply equipment the way they
# charge the track — on the weight moved rather than the path used.
CATENARY_GTKM: dict[str, SV] = {
    "FI": SV(
        "FI",
        "catenary_eur_gross_tonne_km",
        0.000167,
        "EUR/gross-tonne-km",
        SOURCED,
        "FI-NS-2027",
        "Tab.2 §5.3",
        "EUR",
        2027,
        "0.0167 cents per gross-tonne-km. Finland charges everything per "
        "gross-tonne-km, this term included",
    ),
    "GR": SV(
        "GR",
        "catenary_eur_gross_tonne_km",
        0.00210 * 1.1975 * 0.60,
        "EUR/gross-tonne-km",
        DERIVED,
        "GR-OSE-2026",
        "ch.6 c_wte",
        "EUR",
        2026,
        "Electrification wear c_wte 0.00210 EUR/tonne-km x the network "
        "statement's own inflation multiplier 1.1975 x the phased recovery "
        "factor 0.60 = 0.00150885. The multiplier is the document's step from "
        "its 2019 base to 2026 money, so the price basis recorded here is "
        "2026 — recording 2019 would escalate to 2026 twice",
    ),
    "SK": SV(
        "SK",
        "catenary_eur_gross_tonne_km",
        0.000228,
        "EUR/gross-tonne-km",
        SOURCED,
        "SK-ZSR-A52B",
        "Annex 1, component U4",
        "EUR",
        2019,
        "Published as 0.228 EUR per 1,000 gross-tonne-km. Frozen with the rest "
        "of Measure 2/2018, hence the country escalation deviation",
    ),
}

# Belgium charges the catenary per unit of energy, so it is a price component
# and not a separate column — it joins the Belgian working price directly.
SUPPLY_EQUIPMENT_KWH: dict[str, SV] = {
    "BE": SV(
        "BE",
        "supply_equipment_eur_kwh",
        0.01722,
        "EUR/kWh",
        SOURCED,
        "BE-NS-2027",
        "§5.3 direct cost catenary",
        "EUR",
        2026,
        "17.22 EUR/MWh for catenary use — the only supply-equipment charge in "
        "the calibration published per unit of energy rather than per train-km",
    ),
}

# Positively documented absences. Distinct from the MISSING list below, which
# is the honest admission that a document has not been read.
CATENARY_NOT_LEVIED: dict[str, str] = {
    "AT": "The Bahnstromnetz tariff IS the supply-equipment charge and is "
    "already priced per kWh as the network override (mode c); a second "
    "per-train-km term would charge the same asset twice",
    "CH": "The NZV Art.20a traction-current price is all-in; Switzerland "
    "levies no separate charge for the supply installations",
    "DE": "The DB Energie Netznutzung tariff is the supply-equipment charge, "
    "already priced per kWh as the network override (mode c)",
    "IE": "The traction-power charge applies to the DART network only, and "
    "the Irish intercity network a night train would use is unelectrified",
    "SE": "The Trafikverket price is energy plus grid together (0.6075 + "
    "0.1081 SEK/kWh), so the supply equipment is inside the working price",
}

# Charges that demonstrably exist but whose rate could not be read, and
# countries whose position is simply unchecked. Both end up NULL in the
# database and therefore priced at zero, which understates cost — so they are
# recorded as MISSING and listed as open actions rather than quietly defaulted.
CATENARY_MISSING: dict[str, tuple[str, str]] = {
    "BG": (
        "BG-NRIC-2026",
        "Section I lists 17.29 EUR for use of power-supply equipment without "
        "a unit that can be read unambiguously — per train-km, per MWh and "
        "per path are all consistent with the surrounding text",
    ),
    "ES": (
        "ES-BOE-2024",
        "Modality C covers transformation and distribution of traction "
        "electricity; the consulted consolidation names it without a rate table",
    ),
    "UK": (
        "UK-NR-CP7",
        "The Electrification Asset Usage Charge sits in a price-list sheet "
        "not yet extracted",
    ),
    "CZ": ("CZ-NS-2027", "Charging annex not yet checked for a traction-current term"),
    "DK": (
        "DK-NS-2027",
        "Danish charges are set by executive order rather than in the network "
        "statement; the order has not been checked for an electrification term",
    ),
    "EE": ("EE-TTJA-2026", "Published rate table not yet checked"),
    "NL": (
        "NL-NS-2027",
        "Not established whether the train path service price includes "
        "catenary use or whether ProRail charges it separately",
    ),
    "NO": ("NO-NS-2027", "§5.3.3 not yet checked for an electrification term"),
    "PT": (
        "PT-IP-2027",
        "§5.3 distinguishes electric from diesel traction, so a separate "
        "supply-equipment element is plausible and unchecked",
    ),
    "SI": ("SI-NS-2027", "§5.3 factor chain not yet checked"),
}

# The countries whose TAC section records an "excluded (energy)" line. This is
# the handover list from models/infrastructure/tac/calib/TAC_CALIBRATION.md and
# the reason the two domains together neither drop a charge nor count one
# twice: every country here must have a position below, priced or documented.
TAC_ENERGY_EXCLUSIONS = {
    "BE",
    "BG",
    "CH",
    "ES",
    "FI",
    "FR",
    "GR",
    "HR",
    "HU",
    "IE",
    "IT",
    "LT",
    "LU",
    "LV",
    "PL",
    "RO",
    "SK",
    "UK",
}

_covered = (
    set(CATENARY_TRAIN_KM)
    | set(CATENARY_GTKM)
    | set(SUPPLY_EQUIPMENT_KWH)
    | set(CATENARY_NOT_LEVIED)
    | set(CATENARY_MISSING)
)
assert _covered == set(COUNTRIES) - NO_RAILWAY_COUNTRIES, (
    "every railway country needs a supply-equipment position: "
    f"{sorted(set(COUNTRIES) - NO_RAILWAY_COUNTRIES - _covered)} unaccounted for"
)
assert TAC_ENERGY_EXCLUSIONS <= _covered, (
    "a charge TAC excluded as energy has no position here: "
    f"{sorted(TAC_ENERGY_EXCLUSIONS - _covered)}"
)
print(
    f"supply equipment: {len(CATENARY_TRAIN_KM)} per train-km, "
    f"{len(CATENARY_GTKM)} per gross-tonne-km, "
    f"{len(SUPPLY_EQUIPMENT_KWH)} per kWh, "
    f"{len(CATENARY_NOT_LEVIED)} not levied, {len(CATENARY_MISSING)} missing"
)

## Working price assembly

One day price and, where a country bands its tariff, one night price. Each
component is carried to the evaluation year on **its own** price basis and
escalation rate before the sum is taken, so a 2016 statutory tax and a 2027
tariff never get the same treatment just because they land in the same row.

In [ ]:
# --- Working price ---------------------------------------------------------
# The formula, per band:
#
#   mode (b): the all-in tariff for that band
#   mode (a) and (c): sum(non-VAT Eurostat components)
#                     - network column + traction-network tariff   [mode c]
#                     - environmental column + rail electricity tax
#                     + VAT                                        [DK only]
#                     + per-kWh supply equipment                   [BE only]
#
# Every term enters as EUR at TARGET_YEAR. The result is stored as an SV with
# basis TARGET_YEAR so it escalates by nothing further downstream.
BANDS = ("day", "night")


def _price_parts(cc: str, band: str) -> list[tuple[str, SV, int]]:
    """The signed components of one country's working price for one band."""
    if cc in IM_TARIFF:
        sv = IM_TARIFF[cc].get(band) or IM_TARIFF[cc]["day"]
        return [("all-in tariff", sv, +1)]

    comp = eurostat[cc]
    parts: list[tuple[str, SV, int]] = [
        (f"eurostat {name}", comp[name], +1) for name in NON_VAT
    ]

    if cc in NETWORK_OVERRIDE:
        override = NETWORK_OVERRIDE[cc].get(band) or NETWORK_OVERRIDE[cc]["day"]
        parts.append(("eurostat network (replaced)", comp["network"], -1))
        parts.append(("traction network tariff", override, +1))

    tax = RAIL_TAX.get(cc)
    if tax is not None and cc not in NO_TAX_RECONCILIATION:
        parts.append(("eurostat environmental (replaced)", comp["environmental"], -1))
        parts.append(("rail electricity tax", tax, +1))

    if not VAT_RECOVERABLE[cc]:
        parts.append(("non-deductible VAT", comp["vat"], +1))

    if cc in SUPPLY_EQUIPMENT_KWH:
        parts.append(("supply equipment per kWh", SUPPLY_EQUIPMENT_KWH[cc], +1))

    return parts


def _anchor_source(cc: str, band: str) -> tuple[str, str]:
    """The document a working price is filed under: the override or all-in
    tariff where one exists, otherwise Eurostat."""
    if cc in IM_TARIFF:
        sv = IM_TARIFF[cc].get(band) or IM_TARIFF[cc]["day"]
    elif cc in NETWORK_OVERRIDE:
        sv = NETWORK_OVERRIDE[cc].get(band) or NETWORK_OVERRIDE[cc]["day"]
    else:
        sv = eurostat[cc]["energy_supply"]
    return sv.source_id, sv.locator


PRICE_MODES = {
    **{cc: "im_tariff" for cc in IM_TARIFF},
    **{cc: "network_override" for cc in NETWORK_OVERRIDE},
}

price_parts: dict[tuple[str, str], list[tuple[str, SV, int]]] = {}
working: dict[tuple[str, str], SV] = {}

for cc in sorted(set(EUROSTAT_IE) | set(IM_TARIFF)):
    banded = cc in NIGHT_BANDS
    for band in BANDS:
        parameter = f"working_price_{band}"
        if band == "night" and not banded:
            working[(cc, band)] = SV(
                cc,
                parameter,
                None,
                "EUR/kWh",
                NOT_LEVIED,
                *_anchor_source(cc, "day"),
                "EUR",
                None,
                "One rate around the clock — the tariff has no night band",
            )
            continue

        parts = _price_parts(cc, band)
        price_parts[(cc, band)] = parts
        total = sum(sign * sv.model_value for _, sv, sign in parts)
        formula = " ".join(
            f"{'+' if sign > 0 else '-'} {label}" for label, _, sign in parts
        ).lstrip("+ ")
        source_id, locator = _anchor_source(cc, band)
        working[(cc, band)] = SV(
            cc,
            parameter,
            round(total, 6),
            "EUR/kWh",
            DERIVED,
            source_id,
            locator,
            "EUR",
            TARGET_YEAR,
            f"{formula}; every term converted to EUR at the {FX_SNAPSHOT} "
            f"snapshot and carried to {TARGET_YEAR} on its own price basis",
        )

# Plausibility: a European traction price outside 3-40 cents per kWh is an
# arithmetic error, not a tariff.
for (cc, band), sv in working.items():
    if sv.value is not None:
        assert 0.03 <= sv.value <= 0.40, f"{cc} {band} price implausible: {sv.value}"

# A night rate above the day rate would mean the band mechanism is inverted
# somewhere — every banded tariff in Europe discounts the night.
for cc in NIGHT_BANDS:
    day, night = working[(cc, "day")].value, working[(cc, "night")].value
    assert night < day, f"{cc}: night rate {night} not below day rate {day}"

print(
    f"{sum(1 for v in working.values() if v.value is not None)} working prices "
    f"across {len(set(k[0] for k in working))} countries"
)
for cc in sorted(NIGHT_BANDS):
    print(
        f"    {cc}: day {working[(cc, 'day')].value:.4f} / "
        f"night {working[(cc, 'night')].value:.4f} EUR/kWh"
    )

## Export — the calibration record

`data/energy_prices.csv` is the audit trail: every component, in its own
currency at its own basis. `data/energy_price_modes.csv` is the resolved
per-country result at the evaluation year — the same numbers the database
receives, in one row per country.

In [ ]:
# --- Export ----------------------------------------------------------------


def write_csv(path: Path, columns: list[str], rows: list[dict]) -> None:
    with open(path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        for row in rows:
            writer.writerow(
                {c: ("" if row.get(c) is None else row[c]) for c in columns}
            )
    print(f"  {path.name}: {len(rows)} rows")


def write_data(name: str, columns: list[str], rows: list[dict]) -> None:
    write_csv(DATA_DIR / name, columns, rows)


# --- the two catenary columns, resolved for every railway country ---
# Precedence: this unit, then the other unit, then per-kWh, then a documented
# absence, then MISSING. A country charging per train-km carries a documented
# NULL in the gross-tonne-km column rather than an unexplained blank, so the
# two columns together always say which unit the IM chose.
_CATENARY_TABLES = {
    "catenary_eur_train_km": CATENARY_TRAIN_KM,
    "catenary_eur_gross_tonne_km": CATENARY_GTKM,
}

catenary: dict[tuple[str, str], SV] = {}
for cc in set(COUNTRIES) - NO_RAILWAY_COUNTRIES:
    for parameter, table in _CATENARY_TABLES.items():
        other_param, other = next(
            (p, t) for p, t in _CATENARY_TABLES.items() if p != parameter
        )
        if cc in table:
            catenary[(cc, parameter)] = table[cc]
        elif cc in other:
            catenary[(cc, parameter)] = SV(
                cc,
                parameter,
                None,
                "",
                NOT_LEVIED,
                other[cc].source_id,
                other[cc].locator,
                "EUR",
                None,
                f"Charged in the other unit — see {other_param}",
            )
        elif cc in SUPPLY_EQUIPMENT_KWH:
            catenary[(cc, parameter)] = SV(
                cc,
                parameter,
                None,
                "",
                NOT_LEVIED,
                SUPPLY_EQUIPMENT_KWH[cc].source_id,
                SUPPLY_EQUIPMENT_KWH[cc].locator,
                "EUR",
                None,
                "Charged per unit of energy instead, so it sits inside the "
                "working price — see supply_equipment_eur_kwh",
            )
        elif cc in CATENARY_NOT_LEVIED:
            catenary[(cc, parameter)] = SV(
                cc,
                parameter,
                None,
                "",
                NOT_LEVIED,
                _anchor_source(cc, "day")[0],
                "",
                "EUR",
                None,
                CATENARY_NOT_LEVIED[cc],
            )
        else:
            source_id, why = CATENARY_MISSING[cc]
            catenary[(cc, parameter)] = SV(
                cc, parameter, None, "", MISSING, "", "", "EUR", None, why
            )

# The full record: components plus the resolved parameters, so a reader can
# follow a working price back to the Eurostat cell or tariff line it came from.
values = []
for cc in sorted(set(EUROSTAT_IE)):
    values.extend(eurostat[cc][name] for name in COMPONENTS)
for cc, bands in sorted(IM_TARIFF.items()):
    values.extend(bands[b] for b in BANDS if b in bands)
for cc, bands in sorted(NETWORK_OVERRIDE.items()):
    values.extend(bands[b] for b in BANDS if b in bands)
values.extend(RAIL_TAX[cc] for cc in sorted(RAIL_TAX))
values.extend(SUPPLY_EQUIPMENT_KWH[cc] for cc in sorted(SUPPLY_EQUIPMENT_KWH))
values.extend(NIGHT_BANDS[cc] for cc in sorted(NIGHT_BANDS))
values.append(VAT_FLAG)

full_grid: list[SV] = list(values)
for cc in COUNTRIES:
    for parameter in DB_PARAMETERS:
        if cc in NO_RAILWAY_COUNTRIES:
            full_grid.append(
                SV(cc, parameter, None, "", NO_RAILWAY, note="No railway network")
            )
        elif parameter.startswith("working_price"):
            full_grid.append(working[(cc, parameter.split("_")[-1])])
        else:
            full_grid.append(catenary[(cc, parameter)])

_dupes = [
    k
    for k, n in {
        (v.country_code, v.parameter): sum(
            1
            for w in full_grid
            if (w.country_code, w.parameter) == (v.country_code, v.parameter)
        )
        for v in full_grid
    }.items()
    if n > 1
]
assert not _dupes, f"duplicate country/parameter pair: {sorted(_dupes)}"

write_data("energy_prices.csv", SV_FIELDS, [asdict(v) for v in full_grid])

# --- the resolved per-country result ---
MODE_COLUMNS = [
    "country_code",
    "derivation_mode",
    "price_day_eur_kwh",
    "price_night_eur_kwh",
    "night_band_start",
    "night_band_end",
    "catenary_eur_train_km",
    "catenary_eur_gross_tonne_km",
    "vat_recoverable",
    "supply_equipment_status",
]


def _hhmm(minute: int | None) -> str:
    return "" if minute is None else f"{minute // 60:02d}:{minute % 60:02d}:00"


def _model(sv: SV | None) -> float | None:
    return None if sv is None or sv.model_value is None else round(sv.model_value, 8)


mode_rows = []
for cc in sorted(set(EUROSTAT_IE) | set(IM_TARIFF)):
    band = NIGHT_BANDS.get(cc)
    trkm = catenary[(cc, "catenary_eur_train_km")]
    gtkm = catenary[(cc, "catenary_eur_gross_tonne_km")]
    mode_rows.append(
        {
            "country_code": cc,
            "derivation_mode": PRICE_MODES.get(cc, "benchmark"),
            "price_day_eur_kwh": _model(working[(cc, "day")]),
            "price_night_eur_kwh": _model(working[(cc, "night")]),
            "night_band_start": _hhmm(int(band.value)) if band else "",
            "night_band_end": _hhmm(NIGHT_BAND_END_MIN) if band else "",
            "catenary_eur_train_km": _model(trkm),
            "catenary_eur_gross_tonne_km": _model(gtkm),
            "vat_recoverable": VAT_RECOVERABLE[cc],
            "supply_equipment_status": (
                trkm.status if trkm.value is not None else gtkm.status
            ),
        }
    )
write_data("energy_price_modes.csv", MODE_COLUMNS, mode_rows)

by_status: dict[str, int] = {}
for v in full_grid:
    by_status[v.status] = by_status.get(v.status, 0) + 1
print()
for s, n in sorted(by_status.items(), key=lambda kv: -kv[1]):
    print(f"    {s:11} {n:4}")

# A note claiming an inflation or indexation step must name the year it lands
# on, and that year must be the recorded basis — otherwise the escalation
# applies those years a second time. Greece is the live case.
_INFLATION_WORDS = ("inflation", "indexed", "indexation", "escalat")
_suspect = [
    f"{v.country_code}.{v.parameter}"
    for v in values
    if v.value is not None
    and any(w in v.note.lower() for w in _INFLATION_WORDS)
    and f"{v.basis_year}" not in v.note
]
assert not _suspect, (
    "note claims an inflation step but does not name the year it lands on — "
    f"double-counting risk: {_suspect}"
)

# Every source a value leans on must exist in the register written by 01 — a
# dangling source_id is provenance that looks real and is not.
_register_path = DATA_DIR / "sources_register.csv"
if _register_path.exists():
    with open(_register_path, encoding="utf-8") as fh:
        register = {r["source_id"]: r for r in csv.DictReader(fh)}
    cited = {v.source_id for v in full_grid if v.source_id}
    assert cited <= set(register), (
        f"cited but not in the register: {sorted(cited - set(register))}"
    )
    print(f"\n  provenance: {len(cited)} sources cited, all present in the register")
else:
    raise AssertionError(
        "run 01_source_extraction.ipynb first — the provenance check needs its register"
    )

## Seed export

What the database receives: one row per country, EUR at the evaluation year,
no units and no currencies. An empty cell is a NULL, and for the catenary
columns a NULL means *this country does not levy this* — the fallback for a
country with no calibrated price at all is a separate, deliberate decision
made in the loader.

In [ ]:
# --- Seed export -----------------------------------------------------------
SEED_DIR = DATA_DIR.parent / "seed"
SEED_DIR.mkdir(exist_ok=True)

TRACK_ENERGY_COLUMNS = [
    "country_code",
    "track_energy_price_eur_kwh",
    "track_energy_price_night_eur_kwh",
    "track_energy_night_band_start",
    "track_energy_night_band_end",
    "track_energy_catenary_eur_train_km",
    "track_energy_catenary_eur_gross_tonne_km",
    "source_id",
    "change_log",
]

# Eight decimals, matching the widest energy column in db/schema.py: the
# per-gross-tonne-km terms are of order 1e-4, so coarser rounding would
# quantise a real per-country difference into noise.
_SEED_NDIGITS = 8


def _fmt(value: float) -> str:
    """Fixed-point, trailing zeros stripped — no scientific notation, which
    seed.py's CSV reader would parse but nobody can eyeball."""
    return f"{value:.{_SEED_NDIGITS}f}".rstrip("0").rstrip(".")


def _seed_value(sv: SV | None) -> str:
    if sv is None or sv.model_value is None:
        return ""
    return _fmt(sv.model_value)


track_energy_seed = []
for cc in sorted(set(EUROSTAT_IE) | set(IM_TARIFF)):
    band = NIGHT_BANDS.get(cc)
    # One group-level source FK per country (input_params.track_infrastructures
    # .track_energy_price_src), same contract as the TAC group. The primary is
    # the document the price is filed under — the national tariff where one
    # exists, otherwise Eurostat — not whichever id sorts first: a Danish price
    # is a Eurostat figure with a tax and a VAT adjustment on top, and filing
    # it under the tax study would misattribute it.
    primary = _anchor_source(cc, "day")[0]
    others = sorted(
        {
            v.source_id
            for v in full_grid
            if v.country_code == cc and v.source_id and v.value is not None
        }
        - {primary}
    )
    extra = f"Energy price also sourced from {', '.join(others)}." if others else ""
    track_energy_seed.append(
        {
            "country_code": cc,
            "track_energy_price_eur_kwh": _seed_value(working[(cc, "day")]),
            "track_energy_price_night_eur_kwh": _seed_value(working[(cc, "night")]),
            "track_energy_night_band_start": _hhmm(int(band.value)) if band else "",
            "track_energy_night_band_end": _hhmm(NIGHT_BAND_END_MIN) if band else "",
            "track_energy_catenary_eur_train_km": _seed_value(
                catenary[(cc, "catenary_eur_train_km")]
            ),
            "track_energy_catenary_eur_gross_tonne_km": _seed_value(
                catenary[(cc, "catenary_eur_gross_tonne_km")]
            ),
            "source_id": primary,
            "change_log": extra,
        }
    )


# The fallback row. A country the model routes through but the calibration has
# no price for is priced from the European median rather than for free.
#
# Median, not mean: the calibrated spread runs from 0.06 to 0.33 EUR/kWh and a
# mean over that is pulled up by the expensive tail into a figure no European
# market actually charges.
#
# Only the day price is defaulted. A night band and a catenary charge are
# national particularities — roughly half of Europe's IMs levy a
# supply-equipment charge and three band their tariff — and handing an
# uncalibrated country either would invent tariff structure rather than fill a
# gap. That is the same rule the TAC default follows for seat_km and per_stop.
def _median(xs: list[float]) -> float:
    ordered = sorted(xs)
    mid = len(ordered) // 2
    if len(ordered) % 2:
        return ordered[mid]
    return (ordered[mid - 1] + ordered[mid]) / 2.0


_day_prices = [
    working[(cc, "day")].model_value for cc in set(EUROSTAT_IE) | set(IM_TARIFF)
]
default_energy = {
    "country_code": "_default",
    "track_energy_price_eur_kwh": _fmt(_median(_day_prices)),
}
print(
    f"  default day price: median of {len(_day_prices)} calibrated countries "
    f"= {float(default_energy['track_energy_price_eur_kwh']):.4f} EUR/kWh"
)

write_csv(SEED_DIR / "track_energy_default.csv", TRACK_ENERGY_COLUMNS, [default_energy])
write_csv(SEED_DIR / "track_energy.csv", TRACK_ENERGY_COLUMNS, track_energy_seed)

# The source register, reduced to what input_params.sources stores. Only the
# documents actually cited by a value are seeded: an unused register row is a
# research note, not provenance. Descriptions are built the same way the TAC
# export builds them, so a document cited by both domains produces one
# identical row and seed.py can de-duplicate on the description.
SOURCE_SEED_COLUMNS = ["source_id", "source_description", "source_url", "source_date"]

cited_ids = {v.source_id for v in full_grid if v.source_id and v.value is not None}
source_seed = [
    {
        "source_id": sid,
        "source_description": (
            f"{register[sid]['title']} — {register[sid]['publisher']} "
            f"({register[sid]['pub_year']})"
        ),
        "source_url": register[sid]["url_or_file"],
        "source_date": register[sid]["date_accessed"],
    }
    for sid in sorted(cited_ids)
]
write_csv(SEED_DIR / "sources.csv", SOURCE_SEED_COLUMNS, source_seed)

# Every seeded country must carry a day price, and every night band must
# survive the pivot — a silently dropped band would price a whole country at
# its day rate without failing anything.
assert all(r["track_energy_price_eur_kwh"] for r in track_energy_seed)
assert sum(
    1 for r in track_energy_seed if r["track_energy_price_night_eur_kwh"]
) == len(NIGHT_BANDS)
assert sum(
    1
    for r in track_energy_seed
    if r["track_energy_catenary_eur_train_km"]
    or r["track_energy_catenary_eur_gross_tonne_km"]
) == len(CATENARY_TRAIN_KM) + len(CATENARY_GTKM)

## Document generation

`ENERGY_PRICING_CALIBRATION.md` is a notebook output like the CSVs. Prose that
carries judgement lives in the template verbatim; every number, table row and
source line is injected from live notebook state, so the document and the data
it describes cannot drift apart.

In [ ]:
# --- ENERGY_PRICING_CALIBRATION.md generation ------------------------------
from datetime import date

DOC_PATH = DATA_DIR.parent / "ENERGY_PRICING_CALIBRATION.md"

# Illustration only. models/energy/ currently returns a flat 28 kWh/train-km
# placeholder; it is used here to put a per-train-km supply-equipment charge
# and a per-kWh price on one scale, and nowhere else. Nothing seeded depends
# on it — which is exactly why the catenary charges keep their own units.
REFERENCE_KWH_PER_KM = 28.0
REFERENCE_GROSS_T = 600.0  # 1 loco + 10 coaches, the TAC reference train


def _f(v: float | None, nd: int = 4) -> str:
    if v is None:
        return "—"
    if abs(v) < 0.001:
        return f"{v:.6f}".rstrip("0").rstrip(".")
    return f"{v:.{nd}f}"


def _native(sv: SV) -> str:
    """As published: enough decimals to be recognisable in the source
    document, trailing zeros stripped so 110 HUF does not read as 110.0000."""
    if sv.value is None:
        return "—"
    return f"{sv.value:,.6f}".rstrip("0").rstrip(".") + f" {sv.currency}"


# --- Eurostat master table -------------------------------------------------
_rows = []
for cc in sorted(EUROSTAT_IE):
    comp = eurostat[cc]
    total = sum(comp[n].value for n in NON_VAT)
    cells = " | ".join(_f(comp[n].value) for n in COMPONENTS)
    _rows.append(f"| {cc} | {cells} | **{_f(total)}** |")
_eu27 = dict(zip(COMPONENTS, EUROSTAT_EU27))
_rows.append(
    "| *EU-27* | "
    + " | ".join(_f(_eu27[n]) for n in COMPONENTS)
    + f" | *{_f(sum(_eu27[n] for n in NON_VAT))}* |"
)
EUROSTAT_TABLE_ROWS = "\n".join(_rows)
EU27_TOTAL_EXCL_VAT = sum(_eu27[n] for n in NON_VAT)

# --- full override table ---------------------------------------------------
IM_TARIFF_ROWS = "\n".join(
    f"| {cc} | {_native(bands['day'])} | "
    f"{_native(bands['night']) if 'night' in bands else '—'} | "
    f"{_f(bands['day'].eur)} | "
    f"{_f(bands['night'].eur) if 'night' in bands else '—'} | "
    f"{bands['day'].basis_year} | `{bands['day'].source_id}` {bands['day'].locator} |"
    for cc, bands in sorted(IM_TARIFF.items())
)

# --- network override table ------------------------------------------------
NETWORK_OVERRIDE_ROWS = "\n".join(
    f"| {cc} | {_f(eurostat[cc]['network'].value)} | "
    f"{_f(bands['day'].value)} | "
    f"{_f(bands['night'].value) if 'night' in bands else '—'} | "
    f"{bands['day'].status} | {bands['day'].basis_year} | "
    f"`{bands['day'].source_id}` {bands['day'].locator} |"
    for cc, bands in sorted(NETWORK_OVERRIDE.items())
)

# --- rail tax table --------------------------------------------------------
_levied = [cc for cc, sv in RAIL_TAX.items() if sv.value]
RAIL_TAX_ROWS = "\n".join(
    f"| {cc} | {_f(RAIL_TAX[cc].value)} | "
    f"{_f(eurostat[cc]['environmental'].value) if cc in eurostat else '—'} | "
    f"`{RAIL_TAX[cc].source_id}` | {RAIL_TAX[cc].note} |"
    for cc in sorted(_levied)
)
RAIL_TAX_ZERO = ", ".join(sorted(cc for cc, sv in RAIL_TAX.items() if not sv.value))

# --- supply equipment table ------------------------------------------------
_rows = []
for cc in sorted(
    set(CATENARY_TRAIN_KM) | set(CATENARY_GTKM) | set(SUPPLY_EQUIPMENT_KWH)
):
    if cc in CATENARY_TRAIN_KM:
        sv = CATENARY_TRAIN_KM[cc]
        per_km = sv.model_value
        unit = "per train-km"
    elif cc in CATENARY_GTKM:
        sv = CATENARY_GTKM[cc]
        per_km = sv.model_value * REFERENCE_GROSS_T
        unit = "per gross-tonne-km"
    else:
        sv = SUPPLY_EQUIPMENT_KWH[cc]
        per_km = sv.model_value * REFERENCE_KWH_PER_KM
        unit = "per kWh"
    energy_per_km = working[(cc, "day")].model_value * REFERENCE_KWH_PER_KM
    _rows.append(
        f"| {cc} | {_native(sv)} | {unit} | {sv.status} | {sv.basis_year} | "
        f"**{per_km:.4f}** | {per_km / energy_per_km:.0%} | "
        f"`{sv.source_id}` {sv.locator} |"
    )
CATENARY_ROWS = "\n".join(_rows)

CATENARY_NOT_LEVIED_ROWS = "\n".join(
    f"| {cc} | {reason} |" for cc, reason in sorted(CATENARY_NOT_LEVIED.items())
)
CATENARY_MISSING_ROWS = "\n".join(
    f"| {cc} | `{sid}` | {why} |" for cc, (sid, why) in sorted(CATENARY_MISSING.items())
)
N_CATENARY_LEVIED = (
    len(CATENARY_TRAIN_KM) + len(CATENARY_GTKM) + len(SUPPLY_EQUIPMENT_KWH)
)

# --- final working price table --------------------------------------------
_rows = []
for cc in sorted(set(EUROSTAT_IE) | set(IM_TARIFF)):
    day = working[(cc, "day")].model_value
    night = working[(cc, "night")].model_value
    band = NIGHT_BANDS.get(cc)
    trkm = catenary[(cc, "catenary_eur_train_km")].model_value
    gtkm = catenary[(cc, "catenary_eur_gross_tonne_km")].model_value
    per_km = (
        day * REFERENCE_KWH_PER_KM + (trkm or 0.0) + (gtkm or 0.0) * (REFERENCE_GROSS_T)
    )
    # "—" is a documented absence, "?" an unread one: the two must not look
    # alike in the table a reviewer reads, because only one of them is a bug.
    marks = tuple(
        _f(v) if v is not None else ("?" if sv.status == MISSING else "—")
        for v, sv in (
            (trkm, catenary[(cc, "catenary_eur_train_km")]),
            (gtkm, catenary[(cc, "catenary_eur_gross_tonne_km")]),
        )
    )
    window = (
        f"{_hhmm(int(band.value))[:5]}-{_hhmm(NIGHT_BAND_END_MIN)[:5]}" if band else "—"
    )
    _rows.append(
        f"| {cc} | {PRICE_MODES.get(cc, 'benchmark')} | **{_f(day)}** | "
        f"{_f(night)} | {window} | {marks[0]} | {marks[1]} | {per_km:.2f} |"
    )
WORKING_TABLE_ROWS = "\n".join(_rows)

_day_by_cc = {
    cc: working[(cc, "day")].model_value for cc in set(EUROSTAT_IE) | set(IM_TARIFF)
}
CHEAPEST = min(_day_by_cc, key=_day_by_cc.get)
DEAREST = max(_day_by_cc, key=_day_by_cc.get)
MEDIAN_DAY = _median(list(_day_by_cc.values()))
SPREAD_FACTOR = _day_by_cc[DEAREST] / _day_by_cc[CHEAPEST]
EU27_TARGET = escalate(
    EU27_TOTAL_EXCL_VAT - _eu27["environmental"],
    EUROSTAT_BASIS,
    ENERGY_ESCALATION_PER_YEAR,
)

# --- escalation table -----------------------------------------------------
_basis_counts: dict[int, int] = {}
for v in values:
    if v.value is not None and v.unit not in ("flag", "minute of day"):
        rate = escalation_rate(v.country_code, v.parameter)
        if rate == ENERGY_ESCALATION_PER_YEAR:
            _basis_counts[v.basis_year] = _basis_counts.get(v.basis_year, 0) + 1
ESCALATION_TABLE_ROWS = "\n".join(
    f"| {by} | {n} | {TARGET_YEAR - by} | "
    + " | ".join(
        (
            f"**{(1 + r) ** (TARGET_YEAR - by):.3f}**"
            if r == ENERGY_ESCALATION_PER_YEAR
            else f"{(1 + r) ** (TARGET_YEAR - by):.3f}"
        )
        for r in (
            ENERGY_ESCALATION_LOW,
            0.015,
            ENERGY_ESCALATION_PER_YEAR,
            ENERGY_ESCALATION_HIGH,
        )
    )
    + " |"
    for by, n in sorted(_basis_counts.items())
)

_escalatable = [
    v
    for v in values
    if v.value is not None
    and v.unit not in ("flag", "minute of day")
    and escalation_rate(v.country_code, v.parameter) == ENERGY_ESCALATION_PER_YEAR
]
_mean_uplift = sum(
    (1 + ENERGY_ESCALATION_PER_YEAR) ** (TARGET_YEAR - v.basis_year)
    for v in _escalatable
) / len(_escalatable)

ESCALATION_OVERRIDE_ROWS = "\n".join(
    [
        f"| parameter `{p}` | {r:.0%}/yr | {why} |"
        for p, (r, why) in sorted(ESCALATION_OVERRIDE_PARAMETER.items())
    ]
    + [
        f"| country {cc} | {r:.0%}/yr | {why} |"
        for cc, (r, why) in sorted(ESCALATION_OVERRIDE_COUNTRY.items())
    ]
)

# --- FX table -------------------------------------------------------------
_fx_used = sorted({v.currency for v in values if v.currency != "EUR" and v.value})
FX_TABLE_ROWS = "\n".join(
    f"| {cur} | {1 / FX_TO_EUR[cur]:,.3f} {cur} = 1 EUR | {FX_TO_EUR[cur]:.6f} |"
    for cur in _fx_used
)
FX_CURRENCIES = ", ".join(_fx_used)

# --- source tables --------------------------------------------------------
_METHOD_KINDS = {
    "fx_reference",
    "macro_projection",
    "study",
    "advocacy_analysis",
    "tax_authority",
    "official_statistics",
}
_cited = sorted({v.source_id for v in full_grid if v.source_id and v.value is not None})


def _source_row(sid: str) -> str:
    r = register[sid]
    link = r["url_or_file"]
    shown = f"[link]({link})" if link.startswith("http") else f"`{link}`"
    return (
        f"| `{sid}` | {r['title']} | {r['publisher']} | {r['pub_year']} | "
        f"{r['price_basis_year'] or '—'} | {shown} |"
    )


SOURCE_TABLE_ROWS = "\n".join(
    _source_row(sid) for sid in _cited if register[sid]["kind"] not in _METHOD_KINDS
)
METHOD_SOURCE_ROWS = "\n".join(
    _source_row(sid) for sid in _cited if register[sid]["kind"] in _METHOD_KINDS
)
UNCHECKED_SOURCE_ROWS = "\n".join(
    _source_row(sid) for sid in sorted(register) if register[sid]["used"] == "Not used"
)

# --- counts ---------------------------------------------------------------
_status_counts = {
    s: sum(1 for v in full_grid if v.status == s)
    for s in (SOURCED, DERIVED, BENCHMARK, ASSUMED, NOT_LEVIED, MISSING, NO_RAILWAY)
}
print("document inputs assembled:", {k: v for k, v in _status_counts.items() if v})

In [ ]:
SCOPE = r"""
## What belongs here

The **price of the energy a train uses while driving**, and the charges an
infrastructure manager levies for supplying it. Concretely, per country:

- the traction electricity price in EUR/kWh, day and — where the tariff bands
  it — night;
- the charge for using the catenary and the traction power-supply
  installations, in the unit the IM publishes it in.

Everything else energy-adjacent is calibrated elsewhere, and each boundary is
a decision rather than an omission:

| Not here | Where | Why |
|---|---|---|
| Consumption in kWh per train-km | `models/energy/` | A property of the train and the terrain, not of a national tariff. This domain prices a kWh; that domain counts them |
| Stabling and pre-heating energy | facility domain (shunting and parking) | Drawn while standing, not while driving, and priced very differently: DB InfraGO charges 0.2345 EUR/kWh metered Elektrant power and 0.3682 EUR/kWh for 16.7 Hz pre-heating, two to three times the traction price. Those belong against stabled hours, never against train-km |
| Track access itself | `models/infrastructure/tac/` | The minimum access package. Its calibration excludes every supply-equipment charge per country — this document picks up exactly that list |
| Diesel traction | out of scope | The target network is electric throughout. A route needing diesel haulage would need a fuel price and an emissions factor this domain does not carry |

The TAC boundary is the one worth stating twice, because it is the one where a
charge could be lost or counted twice. `TAC_CALIBRATION.md` records an
"excluded (energy)" line for @@N_TAC_EXCLUSIONS@@ countries; every one of them
appears below as a priced charge, a documented absence, or an open action with
the document named. Switzerland is the mirror case: its all-in traction price
covers the supply installations, so nothing separate is charged here — while
its *Haltezuschlag* stays with TAC, since a stop consumes path capacity rather
than energy.
"""

In [ ]:
METHOD = r"""
## How a price is built

Three derivation modes, in the order they take precedence:

| Mode | Applies to | What it does |
|---|---|---|
| **(b) full override** | @@N_IM_TARIFF@@ countries with a published traction-energy tariff | Replaces the whole component stack — the tariff is what the RU pays, taxes and network included |
| **(c) network override** | @@N_NETWORK_OVERRIDE@@ countries running a dedicated traction-current network | Replaces **only** the network column; the current itself is still procured on the market, so commodity and taxes stay on the benchmark |
| **(a) benchmark** | everyone else | Eurostat's nine price components for non-household consumers, band IE |

Then two adjustments apply to modes (a) and (c): the rail-specific electricity
excise replaces Eurostat's standard-rate environmental column, and VAT is
added back wherever the operator cannot deduct it.

### Why band IE

Band IE is 20,000–69,999 MWh a year. A night-train operation drawing
15–25 kWh/train-km over 1–3 million train-km a year lands inside it, and the
UK DESNZ "Large" series is defined identically, so the GB figure is directly
comparable rather than approximately so.

Two independent infrastructure-manager tariffs confirm the band choice, which
matters because it is the single largest methodological lever in the document:
Hungary's published all-in price of 67.4 HUF/kWh is 0.170 EUR against Eurostat
band IE HU at 0.1711, and Sweden's Trafikverket worked example of 0.7156 SEK
is 0.064 EUR against band IE SE at 0.0696. Neither would match band IA
(< 20 MWh/a), which an earlier draft of this calibration used and which
overstated prices by roughly 40–70 %.

### Why the rail tax replaces a column rather than adding to it

Eurostat's "Environmental taxes" category is where the national electricity
excise sits — at the **standard** rate. Germany makes this legible: the
Eurostat column reads 0.0205 EUR/kWh, which is exactly the standard Stromsteuer
of 20.50 EUR/MWh, while rail pays the §9(2) reduced rate of 11.42. Same tax,
different rate, so they must not be added. The rule is therefore

```
working price = Σ(non-VAT components) − environmental + rail electricity tax
```

with one country excepted. **Poland** carries an environmental component of
0.0446 EUR/kWh, around 37 times the Polish electricity excise, so it evidently
bundles certificate-of-origin and cogeneration obligations that are not the
excise at all. Subtracting it would credit Poland with levies it really pays;
splitting it would be a guess. No reconciliation is applied and the
un-reconciled total stands, which overstates rather than understates Polish
energy cost.

### Day and night

Three countries price traction electricity by clock time: Austria's
Bahnstromnetz Niedertarif, Switzerland's reduced NZV rate, and Croatia's NT
tariff. All three discount the window 22:00–06:00, which is most of a night
train's run but not the evening departure or the morning approach.

The night rate is therefore stored as its own column with the band, not folded
into a single blended figure: `calc_energy_price.py` splits each country run
pro rata by the clock minutes it spends inside the band, the same mechanism
`calc_tac.py` uses for the German night rate. Everywhere else the tariff has
one rate around the clock, which the model reads as *no band* — a documented
tariff fact, not a missing band.

### VAT

VAT is only a real cost where the operator's own output is VAT-**exempt**,
since exemption removes the right to deduct input VAT. Zero-rating and reduced
rating both preserve it.

**Denmark is the one confirmed exception**: Danish law exempts passenger
transport, so a Danish operator cannot reclaim input VAT and the DK working
price includes it. Everywhere else input VAT is assumed recoverable. This is
the assumption in the document that carries the most weight — see the closing
section.
"""

In [ ]:
CONVERSIONS = r"""
## Two conversions, both here

A calibrated value is what the source document says: native currency, at the
document's own price basis. Two conversions stand between that and a number
the cost model can use, and both happen exactly once, in this notebook.
`calc_energy_price.py` never sees a currency or a price basis, and neither does
`seed.py`.

### Currency

@@N_FX_CURRENCIES@@ of the calibrated prices publish in a currency other than
the euro (@@FX_CURRENCIES@@), so the FX table is a calibration input in its own
right rather than a formatting detail. ECB reference rates, snapshot
**@@FX_SNAPSHOT@@** — deliberately the same snapshot the TAC calibration pins,
so both infrastructure domains reach EUR on identical terms and a scenario
repins one date rather than two:

| Currency | Reference rate | → EUR factor |
|---|---|---|
@@FX_TABLE_ROWS@@

### Price basis → @@TARGET_YEAR@@

Energy is escalated at **@@ESCALATION_PCT@@ a year**, and this is where the
domain deliberately parts company with track access.

Track access rises around 3 %/yr, about a point a year in real terms, because
Directive 2012/34 Art. 31–32 pushes infrastructure managers towards full cost
recovery against a growing renewal burden. A traded commodity has no such
mechanism. European wholesale power forwards run flat to falling in real terms
beyond 2027 as renewable capacity displaces gas at the margin, so carrying the
2025 commodity price forward at nominal HICP is the neutral choice: constant
real price, nominal drift only.

| Price basis | Values | Years to @@TARGET_YEAR@@ | @@ESC_LOW_PCT@@ | 1.5 % | **@@ESCALATION_PCT@@** | @@ESC_HIGH_PCT@@ |
|---|---|---|---|---|---|---|
@@ESCALATION_TABLE_ROWS@@

The band brackets the two directions the argument fails in: 0 %/yr if real
prices fall as fast as nominal inflation rises, 3 %/yr if grid reinforcement
and levy growth outpace HICP. Applied across the calibration, the escalation
adds a mean **@@MEAN_UPLIFT@@** to the values carried on the European rate.

Two deviations from that rate are documented rather than buried, and a
parameter-level deviation outranks a country-level one — a statutory tax rate
does not move with a national tariff's escalation, whichever country levies it:

| Deviation | Rate | Reason, with its counter-argument |
|---|---|---|
@@ESCALATION_OVERRIDE_ROWS@@
"""

In [ ]:
BODY = r"""
# Part III — The values

## 1. Eurostat band IE components, 2025 (EUR/kWh)

The benchmark layer, reproduced exactly as Eurostat publishes it — nine
components, plus the non-VAT total. Bulgaria's negative "other" is a net
rebate in the 2025 data and is carried through as published. Switzerland and
Great Britain do not report to `nrg_pc_205_c`; Cyprus and Malta have no
railways.

| Country | Energy & supply | Network | VAT | Renewable | Capacity | Environmental | Nuclear | Other | **Total excl. VAT** |
|---|---|---|---|---|---|---|---|---|---|
@@EUROSTAT_TABLE_ROWS@@

## 2. Mode (b) — full national or IM tariff

| Country | Day, native | Night, native | Day → EUR | Night → EUR | Basis | Source |
|---|---|---|---|---|---|---|
@@IM_TARIFF_ROWS@@

Great Britain's rate is the **excl.-CCL** DESNZ series, because rail traction
is CCL-exempt. The gap between DESNZ's two columns (0.5527 p/kWh) is a survey
average across liable and exempt consumers and must not be read as the
statutory CCL rate of 0.775 p/kWh.

Croatia's figures add the renewables levy of 0.0132 EUR/kWh to the NT and VT
tariffs; its network statement prints excise as 0.000000, which supersedes the
CE Delft PPS estimate for Croatia.

## 3. Mode (c) — traction-network override

Austria, Germany and France each charge for use of a traction-current network
physically distinct from the public grid. Only the network column is replaced:

| Country | Eurostat network | Override day | Override night | Status | Basis | Source |
|---|---|---|---|---|---|---|
@@NETWORK_OVERRIDE_ROWS@@

**Austria** publishes a single-part tariff billed on energy drawn, so it
converts to EUR/kWh with no demand assumption: Hochtarif 52.40 EUR/MWh
06:00–22:00, Niedertarif 43.67 EUR/MWh 22:00–06:00.

**Germany** publishes a two-part tariff, so the per-kWh equivalent is
`Arbeitspreis + Leistungspreis / Benutzungsdauer`, where Benutzungsdauer is
annual energy divided by the annual peak quarter-hour load *of the RU's own
fleet*. A night train running ~10 h per night on ~350 nights has ~3,500
running hours a year, and the coincident peak of a loco-hauled fleet is
roughly 1.7–2.0× its average draw, giving 1,750–2,060 h/a — hence the 1,900
h/a estimate. Two consequences matter more than the point estimate. The band
choice is robust: reaching the ≥ 2,500 h/a tariff system would need more than
~4,250 running hours a year, which a night-only operation cannot approach.
And within the < 2,500 h/a band the sensitivity is small, because the
Leistungspreis is only 23.35 EUR/kWa there — 1,500 h/a gives 0.0877 and
2,400 h/a gives 0.0818, a spread of ±4 %.

Germany's statutory levies (KWKG-Umlage, Offshore-Netzumlage, StromNEV
§19(2)) are named but not quantified in the price sheet, and are deliberately
**not** added: they are non-network levies already carried in Eurostat's tax
columns, which this override leaves intact. Those columns sum to 0.0128
EUR/kWh for Germany, the right order of magnitude for the three levies
combined, so substituting only the network column preserves them exactly once.

**France** is derived rather than read, because the 2024 tariff is crisis-era
and SNCF Réseau does not publish 2027 values until December 2026. DRR Annexe
5.1.2 gives the formula:

```
RCTE-A = purchase price × loss rate / (1 − loss rate)
```

Back-solving the published 2024 tariff validates it: 0.02912 × 0.864 / 0.136
= 0.1850 EUR/kWh implied purchase price, against a published 2024 RFE of
0.18653 — a 0.8 % gap, which is the management margin. Applying it forward at
the published 13.4 % loss rate and the Eurostat 2025 French energy component
of 0.0680 gives RCTE-A 0.0105, plus RCTE-B 0.0210 indexed from 0.02003, for a
network override of 0.0315 against the crisis-era 0.04915. RFE itself — the
supply of traction current by SNCF Réseau — is optional and is not used: an
RU may procure energy elsewhere.

## 4. Rail-specific electricity tax

CE Delft 4.K83's accompanying database, sheet `Rail_Energy taxes_level`, is
the only pan-European source giving per-country rail excise rates rather than
plotting them. Values are PPS-adjusted, so nominal rates differ by national
price level — immaterial here, since the largest entry in the table is 1.4 %
of a working price. Austria and Germany are the two exceptions, both used at
nominal because both are independently published (15.00 and 11.42 EUR/MWh).

| Country | Rail rate | Eurostat environmental (replaced) | Source | Note |
|---|---|---|---|---|
@@RAIL_TAX_ROWS@@

Positively not levied, or rail-exempt: @@RAIL_TAX_ZERO@@.

## 5. Electric supply equipment

The other half of the TAC scope decision. @@N_CATENARY_LEVIED@@ infrastructure
managers charge separately for use of the catenary and the traction
power-supply installations, in three different units — which is why the
database carries two columns and one price component rather than one blended
number. Dividing a per-train-km charge by an assumed consumption would bake
`models/energy/`'s placeholder factor into an infrastructure charge and move
every one of these figures the day that model is calibrated.

The last two columns put the charges on one scale for reading only: EUR per
train-km at @@TARGET_YEAR@@ prices (per-gross-tonne-km rates at
@@REFERENCE_GROSS_T@@ t, the per-kWh rate at @@REFERENCE_KWH_PER_KM@@
kWh/train-km), and that as a share of the same country's traction energy cost.

| Country | As published | Unit | Status | Basis | @@TARGET_YEAR@@ EUR/train-km | Share of energy cost | Source |
|---|---|---|---|---|---|---|---|
@@CATENARY_ROWS@@

Greece is the outlier and is worth a second look rather than a correction: at
@@REFERENCE_GROSS_T@@ t its electrification wear works out around a euro per
train-km, a fifth of the Greek traction energy bill. That follows from the
tariff's own shape — OSE charges everything on weight moved, and c_wte is 37 %
of the track-wear rate c_wt it sits beside, so the ratio is internally
consistent. It is a weight-driven charge on a heavy loco-hauled train, not an
arithmetic slip; a lighter consist pays proportionally less.

Positively not levied — each for a different reason, and each worth stating so
a later reader does not "fix" an apparent gap:

| Country | Why |
|---|---|
@@CATENARY_NOT_LEVIED_ROWS@@

Not yet established. These are NULL in the database and therefore priced at
zero, which understates cost — the first open action:

| Country | Document | What is unresolved |
|---|---|---|
@@CATENARY_MISSING_ROWS@@

## 6. Working prices at @@TARGET_YEAR@@

Everything above, converted and escalated. The final column is illustrative
only: traction energy at @@REFERENCE_KWH_PER_KM@@ kWh/train-km plus the
supply-equipment charge for a @@REFERENCE_GROSS_T@@ t train, which is how
these numbers will feel in a route total.

| Country | Mode | Day EUR/kWh | Night EUR/kWh | Band | Catenary €/trkm | Catenary €/gtkm | ≈ €/train-km |
|---|---|---|---|---|---|---|---|
@@WORKING_TABLE_ROWS@@

In the two catenary columns **—** is a documented absence and **?** an unread
one, which is the difference between a country that charges nothing and a
country nobody has checked. The ≈ €/train-km column adds a supply-equipment
charge priced at zero for every **?**, so those rows read low by 3–10 %.

![Working price per country](energy_price_by_country.svg)

*Figure 1 — day working price per country, @@TARGET_YEAR@@ EUR/kWh, sorted.
Bar colour marks the derivation mode. The dashed line is the EU-27 band IE
benchmark carried to @@TARGET_YEAR@@ on the same basis
(@@EU27_TARGET@@ EUR/kWh). Regenerated by this notebook — do not edit.*

**Reading the spread.** Prices run from @@CHEAPEST@@ at @@MIN_PRICE@@ to
@@DEAREST@@ at @@MAX_PRICE@@ EUR/kWh, a factor of @@SPREAD_FACTOR@@, around a
median of @@MEDIAN_DAY@@. Three groups separate cleanly. The Nordics plus
Switzerland and Croatia are cheap for two different reasons — hydro- and
wind-heavy generation in FI, NO and SE, explicit night tariffs in CH and HR.
A broad middle of twenty countries sits within a factor of about 1.7 of each
other, so for most routes energy price is not a discriminating variable. The
expensive tail is where it matters: Germany is there because of its dedicated
16.7 Hz traction-network tariff rather than the commodity price, and Great
Britain because the DESNZ large-user series is simply high. Both are also the
two networks with the highest track access charges, so the cost penalties
compound rather than offset.

Italy and Romania sit on the benchmark by design, not by omission: both
infrastructure managers supply traction current as an explicit market
pass-through, republished monthly or quarterly against a market index, so a
published annex would contain a formula rather than a price. The same applies
to Portugal.

---

# Part IV — What the database receives

`seed/track_energy.csv`, one row per country, all EUR at @@TARGET_YEAR@@:

| Column | Meaning | NULL means |
|---|---|---|
| `track_energy_price_eur_kwh` | Day/base traction price | never NULL for a calibrated country |
| `track_energy_price_night_eur_kwh` | Night-band price | one rate around the clock |
| `track_energy_night_band_start` / `_end` | The band, clock time | no band |
| `track_energy_catenary_eur_train_km` | Supply-equipment charge per train-km | not levied in this unit |
| `track_energy_catenary_eur_gross_tonne_km` | Supply-equipment charge per gross-tonne-km | not levied in this unit |

Plus `seed/track_energy_default.csv`, the fallback row for a country the model
routes through but the calibration has no price for: the **median** day price
of the @@N_CALIBRATED@@ calibrated countries, @@DEFAULT_DAY@@ EUR/kWh. Median
rather than mean — the spread is wide enough that a mean is pulled up by the
expensive tail into a figure no European market charges.

Only the day price is defaulted. A night band and a supply-equipment charge
are national particularities, so handing an uncalibrated country either would
invent tariff structure rather than fill a gap — the same rule the TAC default
follows for the seat-km and per-stop terms.

---

# Part V — Open actions and the assumption that carries the risk

| # | Action | Impact if it moves |
|---|---|---|
| 1 | Resolve the @@N_CATENARY_MISSING@@ unread supply-equipment positions above. Three are known to exist (BG's unit is ambiguous, ES publishes Modality C without a rate, GB's EAUC is in an unextracted sheet); seven are simply unchecked | Each is 3–10 % of that country's energy cost, currently priced at zero |
| 2 | Confirm the VAT treatment of international rail passenger transport for IE and GB — the question is not the rate but whether the supply is zero-rated (deduction preserved) or exempt (deduction lost) | 15–30 % on the affected country |
| 3 | Replace the derived French 2027 network override with the published values, after December 2026 | FR only, likely a few per cent — the derivation reproduces the 2024 tariff to within 1 % |
| 4 | Confirm Croatia's NT/VT band hours, currently assumed 22:00–06:00 | Fractions of a per cent — HR day and night differ by 0.046 EUR/kWh |
| 5 | Resolve Italy's TA3 rate by line class rather than defaulting to the conventional-network figure | Doubles the Italian catenary charge on any high-speed leg |
| 6 | Cross-check Germany's three statutory levies against netztransparenz.de for the delivery year | Confirms or moves DE by ~0.0128 EUR/kWh |

**The one assumption carrying real risk is VAT recoverability.** Every other
assumption here moves a working price by single-digit per cents. This one moves
it by 15–30 % for any country it applies to, because a VAT-exempt operator
cannot deduct input VAT while a zero-rated one can. Denmark is confirmed
exempt; the rest are assumed recoverable, and the most plausible additional
candidates are Ireland and Great Britain, both of which apply 0 % to passenger
transport — but zero-rating normally preserves the deduction right, which is
why they are assumed recoverable here. It is implemented as a per-country
boolean so that a single flag flip absorbs new evidence.
"""

In [ ]:
CALIBRATION_TEMPLATE = r"""# Traction Energy Pricing — Calibration

The price of the electricity a night train draws while driving, and the
charges levied for supplying it, calibrated per country for
@@N_COUNTRIES@@ European countries.

Generated by `02_energy_pricing_calibration.ipynb` on @@GENDATE@@ — do not
edit by hand; re-run the notebooks (01 then 02, top to bottom) to regenerate
this document and the CSVs under `data/` and `seed/`. Calibration last
reviewed end to end @@REVIEWED@@.

Consumption modelling lives in `models/energy/`; this document calibrates
**prices**. Implementation:
`backend/models/infrastructure/energy_pricing/calc_energy_price.py`.

**Provenance at a glance:** @@N_SOURCED@@ values read directly from a named
locator, @@N_DERIVED@@ derived by documented arithmetic, @@N_BENCHMARK@@ from
the Eurostat benchmark, @@N_ASSUMED@@ assumed with a mandatory band, and
@@N_NOT_LEVIED@@ documented as not levied — across @@N_CITED@@ cited sources.
@@N_MISSING@@ are `missing`: explicit rather than absent, so a gap cannot be
mistaken for a zero.

---

# Part I — Scope

@@SCOPE@@

---

# Part II — Method

@@METHOD@@

@@CONVERSIONS@@

---

@@BODY@@

---

## Sources

Documents a calibrated value leans on directly. Every citation is checked
against the register written by `01_source_extraction.ipynb`, so a source that
does not resolve fails the notebook rather than reaching this document.

| source_id | Document | Publisher | Published | Price basis | Link |
|---|---|---|---|---|---|
@@SOURCE_TABLE_ROWS@@

Statistical, method and conversion sources — these do not price one country,
they underpin the benchmark, the tax reconciliation, the FX table and the
escalation rate:

| source_id | Document | Publisher | Published | Price basis | Link |
|---|---|---|---|---|---|
@@METHOD_SOURCE_ROWS@@

Registered but not yet cited — the documents behind open action 1. They are in
the register so the gap is addressable rather than invisible:

| source_id | Document | Publisher | Published | Price basis | Link |
|---|---|---|---|---|---|
@@UNCHECKED_SOURCE_ROWS@@

Stored documents follow `{source_id}.{ext}`, so the filename is derivable from
the register and needs no column of its own.
"""

In [ ]:
# --- render ----------------------------------------------------------------
TOKENS = {
    "GENDATE": date.today().isoformat(),
    "REVIEWED": CALIBRATION_REVIEWED,
    "TARGET_YEAR": str(TARGET_YEAR),
    "N_COUNTRIES": str(len(set(EUROSTAT_IE) | set(IM_TARIFF))),
    "N_CALIBRATED": str(len(_day_prices)),
    "N_SOURCED": str(_status_counts[SOURCED]),
    "N_DERIVED": str(_status_counts[DERIVED]),
    "N_BENCHMARK": str(_status_counts[BENCHMARK]),
    "N_ASSUMED": str(_status_counts[ASSUMED]),
    "N_NOT_LEVIED": str(_status_counts[NOT_LEVIED]),
    "N_MISSING": str(_status_counts[MISSING]),
    "N_CITED": str(len(_cited)),
    "N_IM_TARIFF": str(len(IM_TARIFF)),
    "N_NETWORK_OVERRIDE": str(len(NETWORK_OVERRIDE)),
    "N_TAC_EXCLUSIONS": str(len(TAC_ENERGY_EXCLUSIONS)),
    "N_CATENARY_LEVIED": str(N_CATENARY_LEVIED),
    "N_CATENARY_MISSING": str(len(CATENARY_MISSING)),
    "N_FX_CURRENCIES": str(len(_fx_used)),
    "FX_CURRENCIES": FX_CURRENCIES,
    "FX_SNAPSHOT": FX_SNAPSHOT,
    "FX_TABLE_ROWS": FX_TABLE_ROWS,
    "ESCALATION_PCT": f"{ENERGY_ESCALATION_PER_YEAR:.1%}",
    "ESC_LOW_PCT": f"{ENERGY_ESCALATION_LOW:.1%}",
    "ESC_HIGH_PCT": f"{ENERGY_ESCALATION_HIGH:.1%}",
    "ESCALATION_TABLE_ROWS": ESCALATION_TABLE_ROWS,
    "ESCALATION_OVERRIDE_ROWS": ESCALATION_OVERRIDE_ROWS,
    "MEAN_UPLIFT": f"+{(_mean_uplift - 1) * 100:.0f}%",
    "EUROSTAT_TABLE_ROWS": EUROSTAT_TABLE_ROWS,
    "IM_TARIFF_ROWS": IM_TARIFF_ROWS,
    "NETWORK_OVERRIDE_ROWS": NETWORK_OVERRIDE_ROWS,
    "RAIL_TAX_ROWS": RAIL_TAX_ROWS,
    "RAIL_TAX_ZERO": RAIL_TAX_ZERO,
    "CATENARY_ROWS": CATENARY_ROWS,
    "CATENARY_NOT_LEVIED_ROWS": CATENARY_NOT_LEVIED_ROWS,
    "CATENARY_MISSING_ROWS": CATENARY_MISSING_ROWS,
    "WORKING_TABLE_ROWS": WORKING_TABLE_ROWS,
    "REFERENCE_KWH_PER_KM": f"{REFERENCE_KWH_PER_KM:g}",
    "REFERENCE_GROSS_T": f"{REFERENCE_GROSS_T:g}",
    "CHEAPEST": CHEAPEST,
    "DEAREST": DEAREST,
    "MIN_PRICE": _f(_day_by_cc[CHEAPEST]),
    "MAX_PRICE": _f(_day_by_cc[DEAREST]),
    "MEDIAN_DAY": _f(MEDIAN_DAY),
    "DEFAULT_DAY": _f(float(default_energy["track_energy_price_eur_kwh"])),
    "SPREAD_FACTOR": f"{SPREAD_FACTOR:.1f}",
    "EU27_TARGET": _f(EU27_TARGET),
    "SOURCE_TABLE_ROWS": SOURCE_TABLE_ROWS,
    "METHOD_SOURCE_ROWS": METHOD_SOURCE_ROWS,
    "UNCHECKED_SOURCE_ROWS": UNCHECKED_SOURCE_ROWS,
    "SCOPE": SCOPE.strip(),
    "METHOD": METHOD.strip(),
    "CONVERSIONS": CONVERSIONS.strip(),
    "BODY": BODY.strip(),
}


def _render(text: str) -> str:
    """Substitute repeatedly: the prose blocks carry tokens of their own."""
    for _ in range(3):
        for key, value in TOKENS.items():
            text = text.replace(f"@@{key}@@", value)
        if "@@" not in text:
            break
    return text


document = _render(CALIBRATION_TEMPLATE)

# An unsubstituted token would ship a literal @@NAME@@ into a published
# document; a stale one would leave a number nobody can trace.
_left = sorted({t.split("@@")[0] for t in document.split("@@")[1::2]})
assert not _left, f"unsubstituted tokens: {_left}"

DOC_PATH.write_text(document, encoding="utf-8")
print(
    f"  {DOC_PATH.name}: {len(document.splitlines())} lines, "
    f"{len(document):,} characters"
)

## Figure

Regenerates `energy_price_by_country.svg` next to the document. This cell
imports matplotlib, which is what keeps it out of `db/dev/seed.py`'s
stdlib-only execution path — seeding stops as soon as the seed CSVs exist and
never reaches here.

In [ ]:
# Display and figure only — matplotlib and pandas are fine below this line,
# seed.py skips these cells.
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

MODE_COLOURS = {
    "benchmark": "#4C72B0",
    "network_override": "#C44E52",
    "im_tariff": "#55A868",
}
MODE_LABELS = {
    "benchmark": "Eurostat band IE benchmark",
    "network_override": "Traction-network override",
    "im_tariff": "IM / national all-in tariff",
}

_plot = sorted(_day_by_cc.items(), key=lambda kv: kv[1])
_labels = [cc for cc, _ in _plot]
_prices = [p for _, p in _plot]
_colours = [MODE_COLOURS[PRICE_MODES.get(cc, "benchmark")] for cc in _labels]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(_labels, _prices, color=_colours)
ax.axvline(
    EU27_TARGET,
    color="#444444",
    linestyle="--",
    linewidth=1,
    label=f"EU-27 benchmark {EU27_TARGET:.3f}",
)
for y, (cc, price) in enumerate(_plot):
    night = working[(cc, "night")].model_value
    if night is not None:
        ax.plot([night], [y], marker="|", markersize=14, color="#222222")
ax.set_xlabel(f"EUR/kWh, {TARGET_YEAR} prices")
ax.set_title(f"Traction electricity working price per country, {TARGET_YEAR}")
ax.invert_yaxis()
handles = [plt.Rectangle((0, 0), 1, 1, color=MODE_COLOURS[m]) for m in MODE_LABELS]
handles.append(plt.Line2D([], [], color="#444444", linestyle="--"))
handles.append(plt.Line2D([], [], color="#222222", marker="|", linestyle="none"))
ax.legend(
    handles,
    list(MODE_LABELS.values())
    + [f"EU-27 benchmark {EU27_TARGET:.3f}", "night-band rate"],
    loc="lower right",
    fontsize=9,
)
fig.tight_layout()
fig.savefig(DATA_DIR.parent / "energy_price_by_country.svg", format="svg")
plt.close(fig)
print(f"  energy_price_by_country.svg: {len(_plot)} countries")

pd.DataFrame(mode_rows).set_index("country_code")